# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zuhairsyed123/ML_internship_Assignments/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Our Feature Vector Construction:**
We will query DuckDB to extract our features from the `month=2026-03` partition and content metadata from `dim_content`. We will target pages with at least 100 GSC impressions in March 2026.
The features we build:
1. `imp_current`: Total GSC search impressions in March 2026.
2. `clk_current`: Total GSC search clicks in March 2026.
3. `pos_current`: Average GSC ranking position in March 2026.
4. `ses_current`: Total GA4 sessions in March 2026.
5. `eng_current`: Total GA4 engaged sessions in March 2026.
6. `scroll_current`: Total GA4 scroll events in March 2026.
7. `word_count`: Static text length of the page.
8. `search_volume`: Static monthly search volume for the keyword.
9. `competition`: Static keyword competition score (0.0 to 1.0).

We will handle missing values by filling them with `0` for counts, and `0` (or another neutral flag) for position. We will encode categorical intent if needed.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

# Load Hugging Face token from .env
env_path = "../../.env"
token = None
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            if line.startswith("HF_TOKEN="):
                token = line.split("=", 1)[1].strip()
                break

if not token:
    raise ValueError("HF_TOKEN not found in .env")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{token}')")
con.execute("SET max_memory = '2GB'")
con.execute("SET threads = 2")

REL = 'hf://datasets/FlyRank/internship-warehouse'

print("--- Querying and building the feature vector ---")
feature_query = f"""
    WITH current_month AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_current,
               SUM(gsc_clicks) AS clk_current,
               AVG(gsc_avg_position) AS pos_current,
               SUM(ga4_sessions) AS ses_current,
               SUM(ga4_engaged_sessions) AS eng_current,
               SUM(scroll_events) AS scroll_current
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id, content_hash_id
        HAVING imp_current >= 100
    ),
    next_month AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_future
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT c.client_hash_id AS client_id, c.content_hash_id AS content_id,
           c.imp_current, c.clk_current, c.pos_current, c.ses_current, c.eng_current, c.scroll_current,
           coalesce(n.imp_future, 0) AS imp_future,
           (CASE WHEN coalesce(n.imp_future, 0) < 0.8 * c.imp_current THEN 1 ELSE 0 END) AS is_declining,
           dc.word_count, dc.search_volume, dc.competition, dc.main_intent
    FROM current_month c
    LEFT JOIN next_month n ON c.client_hash_id = n.client_hash_id AND c.content_hash_id = n.content_hash_id
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') dc ON c.content_hash_id = dc.content_hash_id
"""
df = con.sql(feature_query).df()

# Handle missing values
df['pos_current'] = df['pos_current'].fillna(0)
df['ses_current'] = df['ses_current'].fillna(0)
df['eng_current'] = df['eng_current'].fillna(0)
df['scroll_current'] = df['scroll_current'].fillna(0)
df['word_count'] = df['word_count'].fillna(0)
df['search_volume'] = df['search_volume'].fillna(0)
df['competition'] = df['competition'].fillna(0.0)
df['main_intent'] = df['main_intent'].fillna('unknown')

print(f"Extracted feature frame with {len(df):,} rows.")
print(df.head())

--- Querying and building the feature vector ---


Extracted feature frame with 101,441 rows.
                 client_id                content_id  imp_current  \
0  client_62f4a7e64f5e0096  content_f1c085e5ea530266        207.0   
1  client_62f4a7e64f5e0096  content_c4901b51a12b1c3b        218.0   
2  client_62f4a7e64f5e0096  content_e68b30ce53db783f        150.0   
3  client_62f4a7e64f5e0096  content_c43f069ed25dd56e        291.0   
4  client_62f4a7e64f5e0096  content_a77ceb7afcf6ec78        938.0   

   clk_current  pos_current  ses_current  eng_current  scroll_current  \
0          0.0     5.208102          0.0          0.0             0.0   
1          1.0     8.071242          0.0          0.0             0.0   
2          1.0     4.015052          0.0          0.0             0.0   
3          2.0     6.293391          0.0          0.0             0.0   
4          3.0     2.913256          0.0          0.0             0.0   

   imp_future  is_declining  word_count  search_volume  competition  \
0        64.0             1     

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Our Feature Notes:**
1. `imp_current` (Numeric): GSC impressions sum in March 2026. Handled: No nulls (filtered >= 100). Available: Yes, historical window.
2. `clk_current` (Numeric): GSC clicks sum in March 2026. Handled: No nulls. Available: Yes, historical window.
3. `pos_current` (Numeric): Average position in March 2026. Handled: Nulls filled with `0` (indicating no ranking). Available: Yes, historical window.
4. `ses_current` / `eng_current` / `scroll_current` (Numeric): GA4 traffic & engagement indicators in March 2026. Handled: Nulls filled with `0`. Available: Yes, historical window.
5. `word_count` (Numeric): Page text length from dim_content. Handled: Nulls filled with `0`. Available: Yes, static attribute.
6. `search_volume` / `competition` (Numeric): Search volume and keyword competition score. Handled: Nulls filled with `0`. Available: Yes, static metadata.
7. `main_intent` (Categorical): Search query intent. Handled: Nulls filled with `'unknown'`. Available: Yes, static metadata.

In [2]:
# One-hot encode the categorical column 'main_intent'
df_encoded = pd.get_dummies(df, columns=['main_intent'], drop_first=True)
print("Features after one-hot encoding:")
print([col for col in df_encoded.columns if col not in ['client_id', 'content_id', 'imp_future', 'is_declining']])

Features after one-hot encoding:
['imp_current', 'clk_current', 'pos_current', 'ses_current', 'eng_current', 'scroll_current', 'word_count', 'search_volume', 'competition', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Our Leakage Hunt Experiments:**
1. **Honest Model**: Random Forest trained using only historical March 2026 features to predict decline in April 2026.
2. **Leaked Model**: Random Forest where future GSC impressions `imp_future` (the outcome window metric used to define the label) is included as a feature.
3. **Validation comparison**: We compare performance (accuracy, precision) and check feature importances for both models.
4. **Grouped Split vs Random Split**: We train the Honest model using both a standard Random Split and a Grouped Split by `client_id` (using GroupShuffleSplit) to evaluate client generalization.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score

honest_cols = ['imp_current', 'clk_current', 'pos_current', 'ses_current', 'eng_current', 
               'scroll_current', 'word_count', 'search_volume', 'competition']
# Include intent categories
intent_cols = [col for col in df_encoded.columns if col.startswith('main_intent_')]
honest_cols.extend(intent_cols)

X_honest = df_encoded[honest_cols]
X_leaked = df_encoded[honest_cols + ['imp_future']]
y = df_encoded['is_declining']
groups = df_encoded['client_id']

# 1. Random Split - Honest Model vs Leaked Model
X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaked, y, test_size=0.3, random_state=42, stratify=y)

model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_h, y_tr)
model_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_l, y_tr)

pred_h = model_honest.predict(X_te_h)
pred_l = model_leaked.predict(X_te_l)

print("--- Random Split Performance ---")
print(f"Base Rate:             {y_te.mean():.4f}")
print(f"Honest Model Accuracy: {accuracy_score(y_te, pred_h):.4f}")
print(f"Honest Model Precision:{precision_score(y_te, pred_h):.4f}")
print(f"Leaked Model Accuracy: {accuracy_score(y_te, pred_l):.4f} (Near 100%!)")
print(f"Leaked Model Precision:{precision_score(y_te, pred_l):.4f} (Near 100%!)")

importances = pd.Series(model_leaked.feature_importances_, index=honest_cols + ['imp_future']).sort_values(ascending=False)
print("\nLeaked Model Feature Importances (imp_future dominates!):")
print(importances.head(5))

# 2. Grouped Split (by client_id) - Honest Model
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_honest, y, groups))

X_tr_g, X_te_g = X_honest.iloc[train_idx], X_honest.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_g, y_tr_g)
pred_g = model_grouped.predict(X_te_g)

print("\n--- Grouped Split (Client Holdout) Performance ---")
print(f"Grouped Base Rate:       {y_te_g.mean():.4f}")
print(f"Grouped Model Accuracy:  {accuracy_score(y_te_g, pred_g):.4f}")
print(f"Grouped Model Precision: {precision_score(y_te_g, pred_g):.4f}")
print(f"Gap (Random vs Grouped): {accuracy_score(y_te, pred_h) - accuracy_score(y_te_g, pred_g):.4f}")

--- Random Split Performance ---
Base Rate:             0.5175
Honest Model Accuracy: 0.6410
Honest Model Precision:0.6517
Leaked Model Accuracy: 0.9705 (Near 100%!)
Leaked Model Precision:0.9757 (Near 100%!)

Leaked Model Feature Importances (imp_future dominates!):
imp_future     0.426791
imp_current    0.299214
pos_current    0.062475
word_count     0.061066
clk_current    0.040657
dtype: float64



--- Grouped Split (Client Holdout) Performance ---
Grouped Base Rate:       0.4846
Grouped Model Accuracy:  0.5730
Grouped Model Precision: 0.6086
Gap (Random vs Grouped): 0.0681


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

**Our Exclusions & Rationale:**
1. `trend_direction` / `trend_pct` - Excluded because they are computed over the same time window as the label, leaking the decline target.
2. `imp_future` / `clk_future` - Excluded because future performance metrics are not knowable at the prediction moment and cause target leakage.
3. `client_hash_id` / `content_hash_id` - Excluded from model features to prevent the model from memorizing client-specific baseline levels, ensuring generalization.
4. Pre-calculated product flags (such as needs refresh, CTR opportunity, etc. if available) - Excluded to prevent copying the existing rule-based system.

In [4]:
# Verify that the exclusions are respected in honest_cols
assert 'imp_future' not in honest_cols
assert 'trend_direction' not in honest_cols
assert 'client_hash_id' not in honest_cols
print("Exclusion checks passed successfully.")

Exclusion checks passed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.